# Phase 2: NLP-Based Requirement Prioritization using Learning to Rank Algorithms
### Advanced Hyperparameter Tuning (Optuna), GPU Acceleration (CUDA), & Dual-Scenario Benchmarking

Notebook ini membandingkan 4 algoritma pemeringkatan (*Learning to Rank* / LTR) untuk otomatisasi prioritisasi requirement perangkat lunak:
1. **LightGBM Ranker** (LambdaRank)
2. **XGBoost Ranker** (LambdaMART / NDCG ranking)
3. **SVMRank** (Pairwise Linear Support Vector Ranking)
4. **Random Forest Regressor** (Pointwise Baseline)

---
### 🎯 Metodologi & Fitur Utama Eksperimen
- **GPU Acceleration (NVIDIA CUDA / RTX 3050)**:
  - **XGBoost Ranker**: `tree_method="hist"` dan `device="cuda"`.
  - **LightGBM Ranker**: `device="gpu"` / `device="cpu"` (fallback safe).
  - **Random Forest & SVMRank**: Multi-threaded CPU (`n_jobs=-1`).
- **Group-Based Splitting & 0% Data Leakage Audit**:
  - `GroupShuffleSplit` (80% Train, 20% Test) berbasis `project_id` agar 1 proyek utuh masuk ke Train ATAU Test.
  - *Assertion Audit* eksplisit: `len(set(qids_train).intersection(set(qids_test))) == 0`.
- **Optuna Hyperparameter Tuning dengan Composite Objective**:
  - 3-Fold `GroupKFold` Cross-Validation.
  - Objective seimbang: $\text{Composite Score} = 0.5 \cdot \text{NDCG}@5 + 0.5 \cdot \text{Spearman}$.
- **Eksplorasi Skenario Pembentukan Label**:
  - **Skenario A (Feature Fusion)**: Input $X$ menggabungkan Fitur Domain (`value`, `effort`, `risk`, `stakeholder_priority`) + Fitur NLP `tfidf_*`.
  - **Skenario B (Text-Only NLP)**: Input $X$ murni HANYA menggunakan fitur `tfidf_*` (drop 4 fitur domain) untuk menguji kapabilitas representasi semantik teks mentah.
- **Audit Overfitting (Train vs. Test Evaluation)**:
  - Perbandingan metrik (`NDCG@5`, `MAP`, `Spearman`, `Kendall`) pada Data Train dan Data Test secara bersandingan beserta selisih (*gap*)-nya.
- **MLflow Tracking & Artifact Export**:
  - Logging seluruh run ke MLflow Tracking Server.
  - Simpan model `.pkl` ke `results/models_advanced/` dan rangkuman metrik ke `results/*.csv`.

--- 
### Step 1 | Setup Library, CUDA GPU Detection, & MLflow Tracking

In [ ]:
import os
import sys
import warnings
import logging
import joblib
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost
import optuna
from dotenv import load_dotenv
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVC
from mlflow.exceptions import MlflowException
import lightgbm as lgb
import xgboost as xgb

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from IPython.display import display
except ImportError:
    display = print

cuda_available = False
gpu_device_name = "NVIDIA CUDA GPU (XGBoost probe)"
xgb_device = "cpu"
lgb_device = "cpu"

try:
    import torch
    if torch.cuda.is_available():
        cuda_available = True
        gpu_device_name = torch.cuda.get_device_name(0)
except Exception:
    pass

try:
    test_xgb = xgb.XGBRanker(
        objective="rank:ndcg",
        tree_method="hist",
        device="cuda",
        n_estimators=1,
        random_state=42,
    )
    test_xgb.fit(
        np.array([[1, 2], [3, 4]], dtype=np.float32),
        np.array([1, 0]),
        group=np.array([2]),
    )
    xgb_device = "cuda"
    cuda_available = True
except Exception as exc:
    raise RuntimeError(
        "XGBoost gagal melakukan training menggunakan CUDA. "
        "Training dihentikan agar tidak diam-diam berjalan di CPU. "
        "Pastikan XGBoost CUDA-enabled dan driver NVIDIA sesuai."
    ) from exc

try:
    test_lgb = lgb.LGBMRanker(device="gpu", n_estimators=1, random_state=42, verbose=-1)
    test_lgb.fit(np.array([[1, 2], [3, 4]]), np.array([1, 0]), group=np.array([2]))
    lgb_device = "gpu"
except Exception:
    lgb_device = "cpu"

print("=======================================================================")
print("🖥️  DEVICE CONFIGURATION & ACCELERATION AUDIT")
print(f"   CUDA Available : {cuda_available}")
print(f"   Compute Device : {gpu_device_name}")
print(f"   🚀 XGBoost Device : {xgb_device} (tree_method='hist')")
print(f"   🚀 LightGBM Device: {lgb_device}")
print(f"   ⚙️  Random Forest  : Multi-threaded CPU (n_jobs=-1)")
print(f"   ⚙️  SVMRank        : Multi-threaded CPU (n_jobs=-1)")
print("=======================================================================")

load_dotenv()
DEFAULT_MLFLOW_URI = "https://mlflow.smbgarasibmw.my.id"
configured_mlflow_uri = os.getenv("MLFLOW_TRACKING_URI")
mlflow_uri = (
    configured_mlflow_uri.rstrip("/")
    if configured_mlflow_uri and configured_mlflow_uri.startswith(("http://", "https://"))
    else DEFAULT_MLFLOW_URI
)
experiment_name = "Phase_2_Ranking_Algorithm_Advanced_Tuning"

try:
    mlflow.set_tracking_uri(mlflow_uri)
    try:
        mlflow.set_experiment(experiment_name)
    except MlflowException as exc:
        if "deleted experiment" not in str(exc).lower():
            raise
        experiment_name = f"{experiment_name}_v2"
        mlflow.set_experiment(experiment_name)
except Exception as exc:
    raise RuntimeError(
        f"Gagal terhubung ke MLflow Tracking Server '{mlflow_uri}'. "
        "Periksa koneksi atau autentikasi MLFLOW_TRACKING_USERNAME/MLFLOW_TRACKING_PASSWORD."
    ) from exc

print(f"🔗 MLflow Connected to : {mlflow_uri}")
print(f"📁 MLflow Experiment   : {experiment_name}")
print(f"📦 Optuna Version      : {optuna.__version__}")
print(f"📦 LightGBM Version    : {lgb.__version__}")
print(f"📦 XGBoost Version     : {xgb.__version__}")

🖥️  DEVICE CONFIGURATION & ACCELERATION AUDIT
   CUDA Available : True
   Compute Device : NVIDIA CUDA GPU (XGBoost probe)
   🚀 XGBoost Device : cuda (tree_method='hist')
   🚀 LightGBM Device: cpu
   ⚙️  Random Forest  : Multi-threaded CPU (n_jobs=-1)
   ⚙️  SVMRank        : Multi-threaded CPU (n_jobs=-1)
🔗 MLflow Connected to : https://mlflow.smbgarasibmw.my.id
📁 MLflow Experiment   : Phase_2_Ranking_Algorithm_Advanced_Tuning_v2
📦 Optuna Version      : 4.9.0
📦 LightGBM Version    : 4.7.0
📦 XGBoost Version     : 3.2.0


--- 
### Step 2 | Load Dataset & Pemisahan Fitur Domain vs. Fitur NLP Text

- **Target Variable ($y$)**: `rank`, dengan `rank = 1` sebagai prioritas tertinggi. Label internal dibalik per proyek agar nilai label terbesar merepresentasikan prioritas tertinggi bagi algoritma ranker.
- **Query Group ID ($qid$)**: `project_id` (penanda kelompok proyek requirement).
- **Kolom Metadata (WAJIB DIBUANG)**: `project_id`, `rank`, dan `type_encoded`. `rank` digunakan sebagai target, bukan fitur, agar tidak terjadi *data leakage*.
- **Fitur Domain Kuantitatif**: `value`, `effort`, `risk`, `stakeholder_priority`
- **Fitur Teks NLP**: Seluruh kolom berbasis `tfidf_*`.

In [ ]:
DATA_PATH = "dataset/Dataset_Training_TFIDF.csv"
TARGET_COL = "rank"
QID_COL = "project_id"

df = pd.read_csv(DATA_PATH)
df = df.sort_values(by=QID_COL).reset_index(drop=True)

qids = df[QID_COL]

# Ubah rank 1 menjadi label relevansi dengan nilai terbesar.
dense_rank = df.groupby(QID_COL)[TARGET_COL].rank(method="dense", ascending=True)
max_dense_rank = dense_rank.groupby(qids).transform("max")
y = (max_dense_rank - dense_rank).astype(int)
y.name = "relevance"

# LightGBM membutuhkan nilai relevansi yang berurutan dalam setiap query.
for _, group_labels in y.groupby(qids):
    assert set(group_labels) == set(range(group_labels.nunique())), "❌ Error: Label relevance per group tidak kontigu!"

def to_contiguous_relevance(labels):
    """Encode label relevance lokal untuk menjaga kontrak label LightGBM."""
    return pd.Series(
        pd.factorize(np.asarray(labels), sort=True)[0],
        index=getattr(labels, "index", None),
    ).astype(int)

y_cont = df[TARGET_COL].values

COLS_TO_DROP = [
    "project_id",
    "rank",
    "type_encoded",
    TARGET_COL,
    QID_COL,
]

DOMAIN_FEATURES = ["value", "effort", "risk", "stakeholder_priority"]
TFIDF_FEATURES = [col for col in df.columns if col.startswith("tfidf_")]

assert TARGET_COL in df.columns, "❌ Error: Kolom target 'rank' tidak ditemukan!"
assert "rank" not in DOMAIN_FEATURES and "rank" not in TFIDF_FEATURES, "❌ Error: Kolom 'rank' terdeteksi dalam fitur!"
assert all(f in df.columns for f in DOMAIN_FEATURES), "❌ Error: Ada fitur domain yang hilang di DataFrame!"
assert len(TFIDF_FEATURES) > 0, "❌ Error: Kolom TF-IDF tidak ditemukan!"

# Jaga agar dua tampilan fitur tetap sejajar untuk indeks train/test yang sama.
X_fusion = df[DOMAIN_FEATURES + TFIDF_FEATURES].copy()
X_nlp = df[TFIDF_FEATURES].copy()

print("=======================================================================")
print("📊 DATASET INGESTION & FEATURE PARTITION SUMMARY")
print(f"   Total Sampel Requirement : {len(df)}")
print(f"   Total Jumlah Proyek (QID): {df[QID_COL].nunique()} Proyek")
print(f"   Daftar Proyek            : {list(df[QID_COL].unique())}")
print(f"   Fitur Domain (Kuantitatif): {len(DOMAIN_FEATURES)} kolom -> {DOMAIN_FEATURES}")
print(f"   Fitur Teks NLP (TF-IDF)  : {len(TFIDF_FEATURES)} kolom")
print(f"   Dimensi Matriks Skenario A (Fusion)   : {X_fusion.shape}")
print(f"   Dimensi Matriks Skenario B (Text-Only): {X_nlp.shape}")
print("=======================================================================")
display(df[["project_id", "type_encoded"] + DOMAIN_FEATURES + ["rank"] + TFIDF_FEATURES[:5]].head())

📊 DATASET INGESTION & FEATURE PARTITION SUMMARY
   Total Sampel Requirement : 951
   Total Jumlah Proyek (QID): 30 Proyek
   Daftar Proyek            : ['P1', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P2', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P3', 'P30', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9']
   Fitur Domain (Kuantitatif): 4 kolom -> ['value', 'effort', 'risk', 'stakeholder_priority']
   Fitur Teks NLP (TF-IDF)  : 100 kolom
   Dimensi Matriks Skenario A (Fusion)   : (951, 104)
   Dimensi Matriks Skenario B (Text-Only): (951, 100)


,project_id,type_encoded,value,effort,risk,stakeholder_priority,rank,tfidf_accident,tfidf_applicable,tfidf_appointment,tfidf_appropriately,tfidf_archive
0,P1,1,-1.084794,-1.023511,-1.530392,-1.077938,18,0.0,0.0,0.0,0.0,0.0
1,P1,1,-1.084794,-1.023511,-1.530392,-1.077938,18,0.0,0.0,0.0,0.0,0.0
2,P1,1,-1.084794,-1.023511,-1.530392,-1.077938,18,0.0,0.0,0.0,0.0,0.0
3,P1,1,1.249231,0.070151,-0.737689,1.246595,3,0.0,0.0,0.0,0.0,0.0
4,P1,1,0.082219,-1.023511,-1.530392,0.084329,13,0.0,0.0,0.0,0.0,0.0


--- 
### Step 3 | Group-Based Splitting & Data Leakage Audit

Menggunakan `GroupShuffleSplit` (80% Train, 20% Test) berbasis `project_id` agar 1 proyek utuh masuk ke Train ATAU Test (0% overlap).
Blok assertion audit dijalankan untuk membuktikan tidak ada kebocoran data (*data leakage*).

In [ ]:
# Pisahkan berdasarkan proyek agar requirement dari satu proyek tidak berpindah antar bagian data.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_fusion, y, groups=qids))

X_train_fusion = X_fusion.iloc[train_idx].copy()
X_test_fusion = X_fusion.iloc[test_idx].copy()

X_train_nlp = X_nlp.iloc[train_idx].copy()
X_test_nlp = X_nlp.iloc[test_idx].copy()

y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
qids_train, qids_test = qids.iloc[train_idx].copy(), qids.iloc[test_idx].copy()

# Urutkan baris berdasarkan proyek sebelum mengirim ukuran group ke ranker.
train_order = np.argsort(qids_train.values)
X_train_fusion = X_train_fusion.iloc[train_order].reset_index(drop=True)
X_train_nlp = X_train_nlp.iloc[train_order].reset_index(drop=True)
y_train = y_train.iloc[train_order].reset_index(drop=True)
qids_train = qids_train.iloc[train_order].reset_index(drop=True)

test_order = np.argsort(qids_test.values)
X_test_fusion = X_test_fusion.iloc[test_order].reset_index(drop=True)
X_test_nlp = X_test_nlp.iloc[test_order].reset_index(drop=True)
y_test = y_test.iloc[test_order].reset_index(drop=True)
qids_test = qids_test.iloc[test_order].reset_index(drop=True)

group_train = qids_train.value_counts(sort=False).values
group_test = qids_test.value_counts(sort=False).values

train_projects = set(qids_train.unique())
test_projects = set(qids_test.unique())
overlap = train_projects.intersection(test_projects)

assert len(overlap) == 0, f"❌ CRITICAL ERROR: Data Leakage Terdeteksi! Proyek berikut ada di Train & Test: {overlap}"

print("=======================================================================")
print("✅ AUDIT DATA LEAKAGE PASSED (0% OVERLAP BETWEEN TRAIN & TEST)")
print(f"   Train Set ({len(X_train_fusion)} sampel) : {len(train_projects)} Proyek -> {sorted(list(train_projects))}")
print(f"   Test Set  ({len(X_test_fusion)} sampel)  : {len(test_projects)} Proyek -> {sorted(list(test_projects))}")
print(f"   Group Counts Train (samples/project): {list(group_train)}")
print(f"   Group Counts Test  (samples/project): {list(group_test)}")
print("=======================================================================")

✅ AUDIT DATA LEAKAGE PASSED (0% OVERLAP BETWEEN TRAIN & TEST)
   Train Set (832 sampel) : 24 Proyek -> ['P1', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P19', 'P2', 'P20', 'P21', 'P22', 'P24', 'P26', 'P27', 'P28', 'P29', 'P3', 'P4', 'P5', 'P6', 'P8', 'P9']
   Test Set  (119 sampel)  : 6 Proyek -> ['P17', 'P18', 'P23', 'P25', 'P30', 'P7']
   Group Counts Train (samples/project): [np.int64(47), np.int64(8), np.int64(18), np.int64(10), np.int64(19), np.int64(9), np.int64(3), np.int64(20), np.int64(22), np.int64(231), np.int64(27), np.int64(14), np.int64(17), np.int64(10), np.int64(12), np.int64(9), np.int64(18), np.int64(9), np.int64(99), np.int64(146), np.int64(12), np.int64(52), np.int64(10), np.int64(10)]
   Group Counts Test  (samples/project): [np.int64(25), np.int64(20), np.int64(17), np.int64(11), np.int64(22), np.int64(24)]


--- 
### Step 4 | Evaluasi Metrik Pemeringkatan (NDCG@5, MAP, Spearman, Kendall)

Fungsi evaluasi menghitung metrik secara rata-rata di seluruh kelompok proyek:
- **NDCG@5**: Normalized Discounted Cumulative Gain untuk 5 requirement teratas (evaluasi Top-K).
- **MAP (Mean Average Precision)**: Presisi dokumen relevan per proyek (menggunakan median relevansi sebagai threshold).
- **Spearman Rank Correlation ($\rho$)**: Korelasi rank monotonik keseluruhan.
- **Kendall's Tau ($\tau$)**: Rasio concordant vs. discordant pairs.

In [ ]:
def compute_map(y_true, y_pred, threshold=None):
    """Menghitung Mean Average Precision (MAP) per kelompok proyek."""
    if threshold is None:
        threshold = np.median(y_true)

    binary_true = (y_true >= threshold).astype(int)
    if binary_true.sum() == 0:
        return 0.0

    # Dahulukan prediksi dengan skor tertinggi.
    order = np.argsort(-y_pred)
    binary_true = binary_true[order]

    precisions = []
    n_relevant = 0
    for i, rel in enumerate(binary_true):
        if rel == 1:
            n_relevant += 1
            precisions.append(n_relevant / (i + 1))

    return float(np.mean(precisions)) if precisions else 0.0

def evaluate_ranking_metrics(y_true, y_pred, qids, k=5):
    """Evaluasi korelasi urutan dan mutu ranking secara rata-rata di seluruh query group."""
    spearman_list, kendall_list, ndcg_list, map_list = [], [], [], []

    if hasattr(y_true, "values"):
        yt_all = y_true.values
    else:
        yt_all = np.array(y_true)

    if hasattr(qids, "values"):
        qids_all = qids.values
    else:
        qids_all = np.array(qids)

    yp_all = np.array(y_pred)

    for qid in np.unique(qids_all):
        mask = qids_all == qid
        yt = yt_all[mask]
        yp = yp_all[mask]

        if len(yt) > 1:
            rh, _ = spearmanr(yt, yp)
            kt, _ = kendalltau(yt, yp)
            if not np.isnan(rh):
                spearman_list.append(rh)
            if not np.isnan(kt):
                kendall_list.append(kt)

            ndcg = ndcg_score([yt], [yp], k=k)
            ndcg_list.append(ndcg)

            map_val = compute_map(yt, yp)
            map_list.append(map_val)

    return {
        "Spearman": float(np.mean(spearman_list)) if spearman_list else 0.0,
        "Kendall": float(np.mean(kendall_list)) if kendall_list else 0.0,
        f"NDCG_{k}": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
        "MAP": float(np.mean(map_list)) if map_list else 0.0
    }

print("✅ Modul Evaluasi Metrik Pemeringkatan siap digunakan.")

✅ Modul Evaluasi Metrik Pemeringkatan siap digunakan.


--- 
### Step 5 | Transformasi Pairwise untuk SVMRank

SVMRank memodelkan pemeringkatan sebagai klasifikasi biner pada pasangan selisih fitur requirement $(x_i - x_j)$ dalam kelompok query proyek yang sama dengan label $y_{ij} \in \{+1, -1\}$.

In [ ]:
def create_pairwise_data(X_data, y_data, qids_data):
    """Mengubah dataset pointwise menjadi pasangan perbedaan fitur (Pairwise transformation untuk SVMRank)."""
    X_pairs, y_pairs = [], []

    X_mat = X_data.values if hasattr(X_data, "values") else np.array(X_data)
    y_arr = y_data.values if hasattr(y_data, "values") else np.array(y_data)
    q_arr = qids_data.values if hasattr(qids_data, "values") else np.array(qids_data)

    for qid in np.unique(q_arr):
        idx = np.where(q_arr == qid)[0]
        X_group = X_mat[idx]
        y_group = y_arr[idx]

        n_samples = len(idx)
        for i in range(n_samples):
            for j in range(i + 1, n_samples):
                if y_group[i] != y_group[j]:
                    # Satu vektor selisih mewakili item yang lebih diprioritaskan.
                    diff = X_group[i] - X_group[j]
                    label = 1 if y_group[i] > y_group[j] else -1

                    X_pairs.append(diff)
                    y_pairs.append(label)

                    X_pairs.append(-diff)
                    y_pairs.append(-label)

    return np.array(X_pairs), np.array(y_pairs)

print("✅ Helper Transformasi Pairwise SVMRank siap digunakan.")

✅ Helper Transformasi Pairwise SVMRank siap digunakan.


--- 
### Step 6 | Skenario A (Feature Fusion: Domain + TF-IDF) – Optuna Composite Metric Tuning

Pada Skenario A, seluruh 4 algoritma dilatih menggunakan kombinasi Fitur Domain + Fitur TF-IDF.
Optuna mengoptimalkan parameter menggunakan **3-Fold GroupKFold CV** dengan **Composite Metric Balance**:
$$\text{Composite Score} = 0.5 \cdot \text{NDCG}@5 + 0.5 \cdot \text{Spearman}$$

In [ ]:
# Buat fold berkelompok sekali agar dapat digunakan ulang oleh setiap trial Optuna.
gkf = GroupKFold(n_splits=3)
cv_folds_fusion = []
svm_cv_folds_fusion = []

for tr_idx, val_idx in gkf.split(X_train_fusion, y_train, groups=qids_train):
    X_tr, y_tr, qids_tr = X_train_fusion.iloc[tr_idx], y_train.iloc[tr_idx], qids_train.iloc[tr_idx]
    X_va, y_va, qids_va = X_train_fusion.iloc[val_idx], y_train.iloc[val_idx], qids_train.iloc[val_idx]
    group_tr = qids_tr.value_counts(sort=False).values
    cv_folds_fusion.append((X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va))

    X_tr_p, y_tr_p = create_pairwise_data(X_tr, y_tr, qids_tr)
    svm_cv_folds_fusion.append((X_tr_p, y_tr_p, X_va, y_va, qids_va))

results_summary = {}
train_results_summary = {}
trained_models = {}
best_params_all = {}

df_preds = df.iloc[test_idx][["project_id", "rank"]].iloc[test_order].copy().reset_index(drop=True)

display(df_preds.head())
print("✅ 3-Fold GroupKFold CV Folds untuk Skenario A (Fusion) berhasil dikonstruksi.")

,project_id,rank
0,P17,4
1,P17,14
2,P17,4
3,P17,21
4,P17,4


✅ 3-Fold GroupKFold CV Folds untuk Skenario A (Fusion) berhasil dikonstruksi.


In [ ]:
model_name = "LightGBM_Ranker_Fusion"

def objective_lgb_fusion(trial):
    params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1
    }
    if lgb_device == "gpu":
        params["device"] = "gpu"

    composite_scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds_fusion:
        model = lgb.LGBMRanker(**params)
        model.fit(X_tr, to_contiguous_relevance(y_tr), group=group_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        score = 0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"]
        composite_scores.append(score)

    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass

print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_lgb_fusion, n_trials=15)

best_params = study.best_params
best_params.update({"objective": "lambdarank", "metric": "ndcg", "random_state": 42, "verbose": -1, "n_jobs": -1})
if lgb_device == "gpu":
    best_params["device"] = "gpu"

print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")

best_model = lgb.LGBMRanker(**best_params)
best_model.fit(X_train_fusion, to_contiguous_relevance(y_train), group=group_train)

preds_train = best_model.predict(X_train_fusion)
preds_test = best_model.predict(X_test_fusion)

metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)

df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params

try:
    mlflow.log_param("scenario", "A_Feature_Fusion")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.lightgbm.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning LightGBM_Ranker_Fusion ===================
Best CV Composite Score : 0.9719
Best Hyperparameters    : {'n_estimators': 100, 'learning_rate': 0.12139707695554378, 'num_leaves': 19, 'max_depth': 4, 'subsample': 0.6180909155642152, 'colsample_bytree': 0.7301321323053057}

-> Test Spearman: 0.9665 (Train Spearman: 0.9828)
-> Test Kendall: 0.9194 (Train Kendall: 0.9513)
-> Test NDCG_5: 0.9929 (Train NDCG_5: 0.9994)
-> Test MAP: 0.9932 (Train MAP: 0.9968)
🏃 View run LightGBM_Ranker_Fusion at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/054c1135ce714be3b004ae76912cd412
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



In [ ]:
model_name = "XGBoost_Ranker_Fusion"

def objective_xgb_fusion(trial):
    params = {
        "objective": "rank:ndcg",
        "tree_method": "hist",
        "device": xgb_device,
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1
    }

    composite_scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds_fusion:
        model = xgb.XGBRanker(**params)
        model.fit(X_tr, y_tr, group=group_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        score = 0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"]
        composite_scores.append(score)

    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass

print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_xgb_fusion, n_trials=15)

best_params = study.best_params
best_params.update({"objective": "rank:ndcg", "tree_method": "hist", "device": xgb_device, "random_state": 42, "n_jobs": -1})
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")

best_model = xgb.XGBRanker(**best_params)
best_model.fit(X_train_fusion, y_train, group=group_train)

preds_train = best_model.predict(X_train_fusion)
preds_test = best_model.predict(X_test_fusion)

metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)

df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params

try:
    mlflow.log_param("scenario", "A_Feature_Fusion")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.xgboost.log_model(best_model, artifact_path="model")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning XGBoost_Ranker_Fusion ===================
Best CV Composite Score : 0.9439
Best Hyperparameters    : {'n_estimators': 75, 'learning_rate': 0.06470799889081995, 'max_depth': 7, 'subsample': 0.7097939893129923, 'colsample_bytree': 0.8510571781023799}

-> Test Spearman: 0.9492 (Train Spearman: 0.9409)
-> Test Kendall: 0.8833 (Train Kendall: 0.8749)
-> Test NDCG_5: 0.9929 (Train NDCG_5: 0.9886)
-> Test MAP: 0.9888 (Train MAP: 0.9794)
🏃 View run XGBoost_Ranker_Fusion at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/628d547cb79b4fd5a06a4078ca769511
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



In [ ]:
model_name = "SVMRank_Fusion"

def objective_svm_fusion(trial):
    C_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
    loss_val = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])
    dual_val = True if loss_val == "hinge" else False

    composite_scores = []
    for X_tr_p, y_tr_p, X_va, y_va, qids_va in svm_cv_folds_fusion:
        model = LinearSVC(C=C_val, loss=loss_val, dual=dual_val, random_state=42, max_iter=2000)
        model.fit(X_tr_p, y_tr_p)
        preds = model.decision_function(X_va.values)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        score = 0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"]
        composite_scores.append(score)

    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass

print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_svm_fusion, n_trials=10)

best_params = study.best_params
dual_val = True if best_params.get("loss") == "hinge" else False
best_params.update({"random_state": 42, "max_iter": 2000, "dual": dual_val})
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")

X_train_p_fusion, y_train_p_fusion = create_pairwise_data(X_train_fusion, y_train, qids_train)
best_model = LinearSVC(**best_params)
best_model.fit(X_train_p_fusion, y_train_p_fusion)

preds_train = best_model.decision_function(X_train_fusion.values)
preds_test = best_model.decision_function(X_test_fusion.values)

metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)

df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params

try:
    mlflow.log_param("scenario", "A_Feature_Fusion")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning SVMRank_Fusion ===================
Best CV Composite Score : 0.9970
Best Hyperparameters    : {'C': 0.03148911647956861, 'loss': 'hinge'}

-> Test Spearman: 0.9831 (Train Spearman: 0.9898)
-> Test Kendall: 0.9516 (Train Kendall: 0.9689)
-> Test NDCG_5: 1.0000 (Train NDCG_5: 1.0000)
-> Test MAP: 1.0000 (Train MAP: 1.0000)
🏃 View run SVMRank_Fusion at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/cc4d15432dab4805873f3b9eb5a6f0f6
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



In [ ]:
model_name = "Random_Forest_Fusion"

def objective_rf_fusion(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
        "random_state": 42,
        "n_jobs": -1
    }

    composite_scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds_fusion:
        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        score = 0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"]
        composite_scores.append(score)

    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass

print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_rf_fusion, n_trials=15)

best_params = study.best_params
best_params.update({"random_state": 42, "n_jobs": -1})
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")

best_model = RandomForestRegressor(**best_params)
best_model.fit(X_train_fusion, y_train)

preds_train = best_model.predict(X_train_fusion)
preds_test = best_model.predict(X_test_fusion)

metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)

df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params

try:
    mlflow.log_param("scenario", "A_Feature_Fusion")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning Random_Forest_Fusion ===================
Best CV Composite Score : 0.8013
Best Hyperparameters    : {'n_estimators': 50, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 4}

-> Test Spearman: 0.3778 (Train Spearman: 0.6564)
-> Test Kendall: 0.2994 (Train Kendall: 0.5475)
-> Test NDCG_5: 0.7403 (Train NDCG_5: 0.8737)
-> Test MAP: 0.7467 (Train MAP: 0.8757)
🏃 View run Random_Forest_Fusion at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/c7ee8cb1760d4a40b42ace694cd65b7c
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



--- 
### Step 7 | Skenario B (Text-Only NLP: Murni TF-IDF) – Optuna Composite Metric Tuning

Pada Skenario B, 4 fitur kuantitatif domain (`value`, `effort`, `risk`, `stakeholder_priority`) **dihapus sepenuhnya** dari matriks $X$.
Model dilatih murni mengandalkan fitur NLP TF-IDF untuk membuktikan kapabilitas asli pemodelan teks tanpa bantuan angka domain.

In [ ]:
cv_folds_nlp = []
svm_cv_folds_nlp = []

for tr_idx, val_idx in gkf.split(X_train_nlp, y_train, groups=qids_train):
    X_tr, y_tr, qids_tr = X_train_nlp.iloc[tr_idx], y_train.iloc[tr_idx], qids_train.iloc[tr_idx]
    X_va, y_va, qids_va = X_train_nlp.iloc[val_idx], y_train.iloc[val_idx], qids_train.iloc[val_idx]
    group_tr = qids_tr.value_counts(sort=False).values
    cv_folds_nlp.append((X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va))

    X_tr_p, y_tr_p = create_pairwise_data(X_tr, y_tr, qids_tr)
    svm_cv_folds_nlp.append((X_tr_p, y_tr_p, X_va, y_va, qids_va))

print("✅ 3-Fold GroupKFold CV Folds untuk Skenario B (Text-Only NLP) berhasil dikonstruksi.")

✅ 3-Fold GroupKFold CV Folds untuk Skenario B (Text-Only NLP) berhasil dikonstruksi.


In [ ]:
model_name = "LightGBM_Ranker_NLP"

def objective_lgb_nlp(trial):
    params = {
        "objective": "lambdarank", "metric": "ndcg",
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42, "verbose": -1, "n_jobs": -1
    }
    if lgb_device == "gpu":
        params["device"] = "gpu"

    composite_scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds_nlp:
        model = lgb.LGBMRanker(**params)
        model.fit(X_tr, to_contiguous_relevance(y_tr), group=group_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        composite_scores.append(0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"])
    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass

print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_lgb_nlp, n_trials=15)

best_params = study.best_params
best_params.update({"objective": "lambdarank", "metric": "ndcg", "random_state": 42, "verbose": -1, "n_jobs": -1})
if lgb_device == "gpu":
    best_params["device"] = "gpu"
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")

best_model = lgb.LGBMRanker(**best_params)
best_model.fit(X_train_nlp, to_contiguous_relevance(y_train), group=group_train)
preds_train = best_model.predict(X_train_nlp)
preds_test = best_model.predict(X_test_nlp)
metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)
df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params

try:
    mlflow.log_param("scenario", "B_Text_Only_NLP")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.lightgbm.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning LightGBM_Ranker_NLP ===================
Best CV Composite Score : 0.3215
Best Hyperparameters    : {'n_estimators': 75, 'learning_rate': 0.13125830316209655, 'num_leaves': 50, 'max_depth': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481}

-> Test Spearman: -0.0089 (Train Spearman: 0.2079)
-> Test Kendall: -0.0164 (Train Kendall: 0.1706)
-> Test NDCG_5: 0.6141 (Train NDCG_5: 0.6280)
-> Test MAP: 0.6517 (Train MAP: 0.6318)
🏃 View run LightGBM_Ranker_NLP at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/309ccb503b3a4fd884899ffa765cb626
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



In [ ]:
model_name = "XGBoost_Ranker_NLP"

def objective_xgb_nlp(trial):
    params = {
        "objective": "rank:ndcg", "tree_method": "hist", "device": xgb_device,
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42, "n_jobs": -1
    }
    composite_scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds_nlp:
        model = xgb.XGBRanker(**params)
        model.fit(X_tr, y_tr, group=group_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        composite_scores.append(0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"])
    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass
print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_xgb_nlp, n_trials=15)
best_params = study.best_params
best_params.update({"objective": "rank:ndcg", "tree_method": "hist", "device": xgb_device, "random_state": 42, "n_jobs": -1})
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")
best_model = xgb.XGBRanker(**best_params)
best_model.fit(X_train_nlp, y_train, group=group_train)
preds_train = best_model.predict(X_train_nlp)
preds_test = best_model.predict(X_test_nlp)
metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)
df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params
try:
    mlflow.log_param("scenario", "B_Text_Only_NLP")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.xgboost.log_model(best_model, artifact_path="model")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning XGBoost_Ranker_NLP ===================
Best CV Composite Score : 0.3840
Best Hyperparameters    : {'n_estimators': 100, 'learning_rate': 0.08513243777587304, 'max_depth': 7, 'subsample': 0.7015271027064849, 'colsample_bytree': 0.7824309989148452}

-> Test Spearman: -0.1976 (Train Spearman: 0.1135)
-> Test Kendall: -0.1680 (Train Kendall: 0.0916)
-> Test NDCG_5: 0.5784 (Train NDCG_5: 0.6399)
-> Test MAP: 0.6001 (Train MAP: 0.6363)
🏃 View run XGBoost_Ranker_NLP at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/f06290cbadca43ef975632fd46519356
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



In [ ]:
model_name = "SVMRank_NLP"

def objective_svm_nlp(trial):
    C_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
    loss_val = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])
    dual_val = True if loss_val == "hinge" else False
    composite_scores = []
    for X_tr_p, y_tr_p, X_va, y_va, qids_va in svm_cv_folds_nlp:
        model = LinearSVC(C=C_val, loss=loss_val, dual=dual_val, random_state=42, max_iter=2000)
        model.fit(X_tr_p, y_tr_p)
        preds = model.decision_function(X_va.values)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        composite_scores.append(0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"])
    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass
print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_svm_nlp, n_trials=10)
best_params = study.best_params
dual_val = True if best_params.get("loss") == "hinge" else False
best_params.update({"random_state": 42, "max_iter": 2000, "dual": dual_val})
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")
X_train_p_nlp, y_train_p_nlp = create_pairwise_data(X_train_nlp, y_train, qids_train)
best_model = LinearSVC(**best_params)
best_model.fit(X_train_p_nlp, y_train_p_nlp)
preds_train = best_model.decision_function(X_train_nlp.values)
preds_test = best_model.decision_function(X_test_nlp.values)
metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)
df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params
try:
    mlflow.log_param("scenario", "B_Text_Only_NLP")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning SVMRank_NLP ===================
Best CV Composite Score : 0.3575
Best Hyperparameters    : {'C': 0.00541524411940254, 'loss': 'squared_hinge'}

-> Test Spearman: 0.0265 (Train Spearman: 0.1435)
-> Test Kendall: 0.0186 (Train Kendall: 0.1071)
-> Test NDCG_5: 0.5598 (Train NDCG_5: 0.6416)
-> Test MAP: 0.6765 (Train MAP: 0.6591)
🏃 View run SVMRank_NLP at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/3ebe05c6ba4448159bc00eedf76df343
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



In [ ]:
model_name = "Random_Forest_NLP"

def objective_rf_nlp(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
        "random_state": 42,
        "n_jobs": -1
    }
    composite_scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds_nlp:
        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        composite_scores.append(0.5 * metrics["NDCG_5"] + 0.5 * metrics["Spearman"])
    return float(np.mean(composite_scores))

try:
    mlflow.start_run(run_name=model_name)
except Exception:
    pass
print(f"=================== Optuna Tuning {model_name} ===================")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_rf_nlp, n_trials=15)
best_params = study.best_params
best_params.update({"random_state": 42, "n_jobs": -1})
print(f"Best CV Composite Score : {study.best_value:.4f}")
print(f"Best Hyperparameters    : {study.best_params}\n")
best_model = RandomForestRegressor(**best_params)
best_model.fit(X_train_nlp, y_train)
preds_train = best_model.predict(X_train_nlp)
preds_test = best_model.predict(X_test_nlp)
metrics_train = evaluate_ranking_metrics(y_train, preds_train, qids_train, k=5)
metrics_test = evaluate_ranking_metrics(y_test, preds_test, qids_test, k=5)
df_preds[f"pred_{model_name}"] = preds_test
train_results_summary[model_name] = metrics_train
results_summary[model_name] = metrics_test
trained_models[model_name] = best_model
best_params_all[model_name] = study.best_params
try:
    mlflow.log_param("scenario", "B_Text_Only_NLP")
    mlflow.log_params(best_params)
    for m_k, m_v in metrics_test.items():
        mlflow.log_metric(f"test_{m_k}", m_v)
        mlflow.log_metric(f"train_{m_k}", metrics_train[m_k])
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
        except Exception:
            pass
    mlflow.end_run()
except Exception:
    for m_k, m_v in metrics_test.items():
        print(f"-> Test {m_k}: {m_v:.4f} (Train {m_k}: {metrics_train[m_k]:.4f})")
print()

=================== Optuna Tuning Random_Forest_NLP ===================
Best CV Composite Score : 0.3038
Best Hyperparameters    : {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 1}

-> Test Spearman: -0.0142 (Train Spearman: 0.1936)
-> Test Kendall: -0.0157 (Train Kendall: 0.1589)
-> Test NDCG_5: 0.5918 (Train NDCG_5: 0.6364)
-> Test MAP: 0.6685 (Train MAP: 0.6643)
🏃 View run Random_Forest_NLP at: https://mlflow.smbgarasibmw.my.id/#/experiments/5/runs/1cd5f9893eb1419ea8e56f6a915cea36
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/5



--- 
### Step 8 | Audit Overfitting (Train vs. Test Evaluation)

Tabel komparasi performa metrik evaluasi (`NDCG@5`, `MAP`, `Spearman`, `Kendall`) pada **Data Train** dan **Data Test** secara bersandingan beserta selisih (*gap*)-nya untuk mendeteksi indikasi memorisasi/overfitting:
- **Formula Gap**: $\text{Gap} = \text{Train Metric} - \text{Test Metric}$
- **Kategori Generalisasi Model**:
  - `Good Generalization`: Gap NDCG@5 $\le 0.08$ dan Gap Spearman $\le 0.08$
  - `Moderate Overfit`: Gap NDCG@5 $\le 0.18$ dan Gap Spearman $\le 0.18$
  - `Severe Overfitting (Memorisasi)`: Gap $> 0.18$ atau model gagal generalisasi ke data uji.

In [ ]:
df_test = pd.DataFrame(results_summary).T
df_train = pd.DataFrame(train_results_summary).T

comp_rows = []
for model in df_test.index:
    tr_ndcg = df_train.loc[model, "NDCG_5"]
    te_ndcg = df_test.loc[model, "NDCG_5"]
    gap_ndcg = tr_ndcg - te_ndcg

    tr_spear = df_train.loc[model, "Spearman"]
    te_spear = df_test.loc[model, "Spearman"]
    gap_spear = tr_spear - te_spear

    tr_map = df_train.loc[model, "MAP"]
    te_map = df_test.loc[model, "MAP"]
    gap_map = tr_map - te_map

    tr_kend = df_train.loc[model, "Kendall"]
    te_kend = df_test.loc[model, "Kendall"]
    gap_kend = tr_kend - te_kend

    if gap_ndcg <= 0.08 and gap_spear <= 0.08:
        status = "Good Generalization"
    elif gap_ndcg <= 0.18 and gap_spear <= 0.18:
        status = "Moderate Overfit"
    else:
        status = "Severe Overfitting (Memorisasi)"

    scenario = "Skenario A (Fusion)" if "Fusion" in model else "Skenario B (Text-Only NLP)"

    comp_rows.append({
        "Scenario": scenario,
        "Model": model,
        "Train_NDCG_5": tr_ndcg,
        "Test_NDCG_5": te_ndcg,
        "Gap_NDCG_5": gap_ndcg,
        "Train_Spearman": tr_spear,
        "Test_Spearman": te_spear,
        "Gap_Spearman": gap_spear,
        "Train_MAP": tr_map,
        "Test_MAP": te_map,
        "Gap_MAP": gap_map,
        "Train_Kendall": tr_kend,
        "Test_Kendall": te_kend,
        "Overfit_Status": status
    })

df_train_vs_test = pd.DataFrame(comp_rows).set_index(["Scenario", "Model"])

print("=========================================================================================================")
print("🔍 AUDIT OVERFITTING: TRAIN VS. TEST RANKING METRIC COMPARISON & GAP ANALYSIS")
print("=========================================================================================================")
display(df_train_vs_test)

🔍 AUDIT OVERFITTING: TRAIN VS. TEST RANKING METRIC COMPARISON & GAP ANALYSIS


Train_NDCG_5  Test_NDCG_5  \
Scenario                   Model                                               
Skenario A (Fusion)        LightGBM_Ranker_Fusion      0.999435     0.992912   
                           XGBoost_Ranker_Fusion       0.988601     0.992912   
                           SVMRank_Fusion              1.000000     1.000000   
                           Random_Forest_Fusion        0.873670     0.740324   
Skenario B (Text-Only NLP) LightGBM_Ranker_NLP         0.628027     0.614119   
                           XGBoost_Ranker_NLP          0.639868     0.578408   
                           SVMRank_NLP                 0.641618     0.559797   
                           Random_Forest_NLP           0.636449     0.591806   

                                                   Gap_NDCG_5  Train_Spearman  \
Scenario                   Model                                                
Skenario A (Fusion)        LightGBM_Ranker_Fusion    0.006523        0.982806   
                           XGBoost_Ranker_Fusion    -0.004311        0.940898   
                           SVMRank_Fusion            0.000000        0.989805   
                           Random_Forest_Fusion      0.133347        0.656393   
Skenario B (Text-Only NLP) LightGBM_Ranker_NLP       0.013909        0.207904   
                           XGBoost_Ranker_NLP        0.061460        0.113529   
                           SVMRank_NLP               0.081821        0.143465   
                           Random_Forest_NLP         0.044643        0.193601   

                                                   Test_Spearman  \
Scenario                   Model                                   
Skenario A (Fusion)        LightGBM_Ranker_Fusion       0.966469   
                           XGBoost_Ranker_Fusion        0.949191   
                           SVMRank_Fusion               0.983078   
                           Random_Forest_Fusion         0.377804   
Skenario B (Text-Only NLP) LightGBM_Ranker_NLP         -0.008900   
                           XGBoost_Ranker_NLP          -0.197562   
                           SVMRank_NLP                  0.026467   
                           Random_Forest_NLP           -0.014210   

                                                   Gap_Spearman  Train_MAP  \
Scenario                   Model                                             
Skenario A (Fusion)        LightGBM_Ranker_Fusion      0.016337   0.996828   
                           XGBoost_Ranker_Fusion      -0.008293   0.979424   
                           SVMRank_Fusion              0.006727   1.000000   
                           Random_Forest_Fusion        0.278590   0.875661   
Skenario B (Text-Only NLP) LightGBM_Ranker_NLP         0.216804   0.631781   
                           XGBoost_Ranker_NLP          0.311091   0.636331   
                           SVMRank_NLP                 0.116997   0.659097   
                           Random_Forest_NLP           0.207811   0.664269   

                                                   Test_MAP   Gap_MAP  \
Scenario                   Model                                        
Skenario A (Fusion)        LightGBM_Ranker_Fusion  0.993164  0.003663   
                           XGBoost_Ranker_Fusion   0.988762 -0.009339   
                           SVMRank_Fusion          1.000000  0.000000   
                           Random_Forest_Fusion    0.746702  0.128960   
Skenario B (Text-Only NLP) LightGBM_Ranker_NLP     0.651701 -0.019920   
                           XGBoost_Ranker_NLP      0.600115  0.036216   
                           SVMRank_NLP             0.676545 -0.017448   
                           Random_Forest_NLP       0.668511 -0.004242   

                                                   Train_Kendall  \
Scenario                   Model                                   
Skenario A (Fusion)        LightGBM_Ranker_Fusion       0.951328   
                           XGBoost_Ranker_

--- 
### Step 9 | Analisis Komparatif Skenario: Feature Fusion vs. Text-Only NLP

Menelaah kontribusi fitur domain kuantitatif terhadap kapabilitas murni representasi teks NLP pada seluruh algoritma.

In [ ]:
print("=======================================================================")
print("📊 HASIL EKSPERIMEN PHASE 2: RANKING ALGORITHM TEST BENCHMARK")
print("=======================================================================")
display(df_test.sort_values(by="NDCG_5", ascending=False))

fusion_models = [m for m in df_test.index if "Fusion" in m]
nlp_models = [m for m in df_test.index if "NLP" in m]

avg_fusion_ndcg = df_test.loc[fusion_models, "NDCG_5"].mean()
avg_fusion_spear = df_test.loc[fusion_models, "Spearman"].mean()
avg_nlp_ndcg = df_test.loc[nlp_models, "NDCG_5"].mean()
avg_nlp_spear = df_test.loc[nlp_models, "Spearman"].mean()

print(f"\n📈 Rata-Rata Skenario A (Feature Fusion) : NDCG@5 = {avg_fusion_ndcg:.4f} | Spearman = {avg_fusion_spear:.4f}")
print(f"📉 Rata-Rata Skenario B (Text-Only NLP)  : NDCG@5 = {avg_nlp_ndcg:.4f} | Spearman = {avg_nlp_spear:.4f}")
print(f"💡 Pengaruh Fitur Domain (Delta A - B)   : Δ NDCG@5 = {(avg_fusion_ndcg - avg_nlp_ndcg):.4f} | Δ Spearman = {(avg_fusion_spear - avg_nlp_spear):.4f}")

📊 HASIL EKSPERIMEN PHASE 2: RANKING ALGORITHM TEST BENCHMARK


,Spearman,Kendall,NDCG_5,MAP
SVMRank_Fusion,0.983078,0.951557,1.000000,1.000000
LightGBM_Ranker_Fusion,0.966469,0.919431,0.992912,0.993164
XGBoost_Ranker_Fusion,0.949191,0.883266,0.992912,0.988762
Random_Forest_Fusion,0.377804,0.299398,0.740324,0.746702
LightGBM_Ranker_NLP,-0.008900,-0.016396,0.614119,0.651701
Random_Forest_NLP,-0.014210,-0.015667,0.591806,0.668511
XGBoost_Ranker_NLP,-0.197562,-0.167964,0.578408,0.600115
SVMRank_NLP,0.026467,0.018630,0.559797,0.676545



📈 Rata-Rata Skenario A (Feature Fusion) : NDCG@5 = 0.9315 | Spearman = 0.8191
📉 Rata-Rata Skenario B (Text-Only NLP)  : NDCG@5 = 0.5860 | Spearman = -0.0486
💡 Pengaruh Fitur Domain (Delta A - B)   : Δ NDCG@5 = 0.3455 | Δ Spearman = 0.8677


--- 
### Step 10 | Ekspor Hasil Eksperimen, CSV Ringkasan, & Model Pickle (.pkl)

Seluruh artefak hasil komparasi disimpan ke folder `results/`:
1. `results/phase_2_advanced_model_comparison.csv`
2. `results/phase_2_train_vs_test_comparison.csv`
3. `results/phase_2_advanced_predictions.csv`
4. `results/best_ranking_model_advanced.pkl`
5. `results/models_advanced/<model_name>.pkl`

In [ ]:
os.makedirs("results/models_advanced", exist_ok=True)

# Simpan tabel perbandingan, prediksi, dan model yang sudah diserialisasi.
df_train_vs_test.to_csv("results/phase_2_train_vs_test_comparison.csv")
df_test.to_csv("results/phase_2_advanced_model_comparison.csv", index_label="Model")
df_preds.to_csv("results/phase_2_advanced_predictions.csv", index=False)

print("💾 Menyimpan semua file pickle (.pkl) model advanced ke 'results/models_advanced/':")
for name, m in trained_models.items():
    model_path = f"results/models_advanced/{name}.pkl"
    joblib.dump(m, model_path)
    print(f"   - Model '{name}' -> {model_path}")

best_model_name = df_test["NDCG_5"].idxmax()
best_ndcg = df_test.loc[best_model_name, "NDCG_5"]
best_spear = df_test.loc[best_model_name, "Spearman"]

joblib.dump(trained_models[best_model_name], "results/best_ranking_model_advanced.pkl")

print("\n=======================================================================")
print(f"🏆 BEST RANKING ALGORITHM : {best_model_name}")
print(f"   Skor Test NDCG@5       : {best_ndcg:.4f}")
print(f"   Skor Test Spearman     : {best_spear:.4f}")
print(f"   Parameter Terbaik      : {best_params_all.get(best_model_name)}")
print(f"   Model Path             : results/best_ranking_model_advanced.pkl")
print("=======================================================================")
print("\n✅ Semua file hasil eksekusi berhasil disimpan di folder 'results/':")
print("   1. results/phase_2_advanced_model_comparison.csv")
print("   2. results/phase_2_train_vs_test_comparison.csv")
print("   3. results/phase_2_advanced_predictions.csv")
print("   4. results/best_ranking_model_advanced.pkl")
print("   5. results/models_advanced/")

💾 Menyimpan semua file pickle (.pkl) model advanced ke 'results/models_advanced/':
   - Model 'LightGBM_Ranker_Fusion' -> results/models_advanced/LightGBM_Ranker_Fusion.pkl
   - Model 'XGBoost_Ranker_Fusion' -> results/models_advanced/XGBoost_Ranker_Fusion.pkl
   - Model 'SVMRank_Fusion' -> results/models_advanced/SVMRank_Fusion.pkl
   - Model 'Random_Forest_Fusion' -> results/models_advanced/Random_Forest_Fusion.pkl
   - Model 'LightGBM_Ranker_NLP' -> results/models_advanced/LightGBM_Ranker_NLP.pkl
   - Model 'XGBoost_Ranker_NLP' -> results/models_advanced/XGBoost_Ranker_NLP.pkl
   - Model 'SVMRank_NLP' -> results/models_advanced/SVMRank_NLP.pkl
   - Model 'Random_Forest_NLP' -> results/models_advanced/Random_Forest_NLP.pkl

🏆 BEST RANKING ALGORITHM : SVMRank_Fusion
   Skor Test NDCG@5       : 1.0000
   Skor Test Spearman     : 0.9831
   Parameter Terbaik      : {'C': 0.03148911647956861, 'loss': 'hinge'}
   Model Path             : results/best_ranking_model_advanced.pkl

✅ Semua fil